# Synthetic demo (selector)

Run one estimator on synthetic data and compare to the ground-truth dose-response curve `h*(t)`.
Choose settings in the **Run selector** cell at the bottom.

Key settings:
- `ALGO`: which method to run (`"ours"`, `"plugin"`, or `"direct"`)
- `N_SAMPLES`: sample size per Monte Carlo run
- `K_RUNS`: number of Monte Carlo runs
- `NOISE_STD`: standard deviation of additive outcome noise
- `FIRST_SEED`: RNG seed for the first run (each run increments by 1)

Methods:
- **Ours (joint-kernel two-stage)**
- **Plug-in (joint-kernel plug-in)**
- **Direct regression baseline (T-only 1D KRR)**


In [1]:
import sys
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Environment + path setup ---
# In Colab, mount Drive and use the repo stored in Google Drive.
# Locally, use the current working directory as the repo root.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = pathlib.Path("/content/drive/MyDrive/Colab Notebooks/CTE_Codes")
except ImportError:
    BASE_DIR = pathlib.Path(".").resolve()

# Make local modules importable (repo root + KRR_methods).
sys.path.append(str(BASE_DIR))
sys.path.append(str(BASE_DIR / "KRR_methods"))

# Project imports: synthetic DGPs and estimators.
from KRR_methods.synthetic_dgps import (
    generate_unified_data,
    approximate_h_star,
)

print(f"Working Directory: {BASE_DIR}")

# Output directory for CSV summaries.
RESULTS_DIR = BASE_DIR / "KRR_methods" / "Results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("RESULTS_DIR =", RESULTS_DIR)


Mounted at /content/drive
Working Directory: /content/drive/MyDrive/Colab Notebooks/CTE_Codes
RESULTS_DIR = /content/drive/MyDrive/Colab Notebooks/CTE_Codes/KRR_methods/Results


In [2]:
# --- Common helpers ---
# Treatment support and evaluation grid for h(t).
T_MIN, T_MAX = -np.pi, np.pi
T_GRID = np.linspace(T_MIN, T_MAX, 1000)

def compute_h_star(t_grid, n_mc_samples=None):
    """Compute ground-truth h*(t) on a grid.

    Args:
        t_grid (np.ndarray): 1D array of treatment values.
        n_mc_samples (int or None): Monte Carlo samples for approximating h*(t)
            if the underlying helper supports it.

    Returns:
        np.ndarray: h*(t) evaluated on t_grid.
    """
    if n_mc_samples is None:
        return approximate_h_star(t_grid)
    try:
        return approximate_h_star(t_grid, n_mc_samples=n_mc_samples)
    except TypeError:
        return approximate_h_star(t_grid)

def save_results_csv(results, prefix):
    """Save per-seed curves and summary statistics to CSV.

    Args:
        results (dict): Output from run_* functions. Expected keys include
            't_grid', 'curves_mat', 'mean_curve', 'se_curve', 'h_star',
            'seeds', and 'run_tag'.
        prefix (str): Kept for compatibility; filename uses results['run_tag'].

    Returns:
        pathlib.Path: Path to the saved CSV file.
    """
    run_tag = results["run_tag"]
    output_path = RESULTS_DIR / f"{run_tag}.csv"

    df_out = pd.DataFrame({"t": results["t_grid"]})
    for i, seed in enumerate(results["seeds"]):
        df_out[f"h_hat_seed_{seed}"] = results["curves_mat"][i, :]

    df_out["mean_h_hat"] = results["mean_curve"]
    df_out["se_h_hat"] = results["se_curve"]
    df_out["h_star"] = results["h_star"]

    df_out.to_csv(output_path, index=False)
    print(f"✅ Saved: {output_path}")
    return output_path


## Algorithm wrappers

Each `run_*` function below runs `K` Monte Carlo replications and returns:
- estimated curves on `T_GRID` for each seed
- mean curve and standard error across runs
- MISE against the ground-truth `h*(t)`


In [3]:
# --- Ours (joint-kernel two-stage) ---
# Hyperparameters for the joint kernel (X, T) and the 1D h(t) smoother.
OURS_KERNEL_TYPE_ND = "matern"
OURS_LENGTH_SCALE_ND = 3.0
OURS_NU_ND = 1.5

OURS_LENGTH_SCALE_1D_H = 3.0
OURS_NU_1D_H = 2.5

# Candidate ridge penalties for the second-stage KRR.
OURS_C_VAL = 0.1
OURS_BETA_H_GRID = np.array([OURS_C_VAL * (2**i) for i in range(0, 9)], dtype=float)
OURS_BETA0_F = OURS_C_VAL
OURS_BETA0_PRIME_F = OURS_C_VAL

from KRR_methods.algorithms.estimators_ours import run_single_ours_synthetic

def run_ours(K=100, n_samples=1000, noise_std=1.0, first_seed=0, n_mc_hstar=40000, prefix="ours_synth"):
    """Run K Monte Carlo replications of the proposed two-stage estimator.

    Args:
        K (int): Number of Monte Carlo runs.
        n_samples (int): Sample size per run.
        noise_std (float): Standard deviation of additive outcome noise.
        first_seed (int): RNG seed for the first run (subsequent runs increment).
        n_mc_hstar (int): MC samples for approximating h*(t) on the grid.
        prefix (str): Label used in output filenames.

    Returns:
        dict: Curves, MISE values, and summary statistics.
    """
    # Ground-truth curve used to compute MISE for each run.
    h_star_vals = compute_h_star(T_GRID, n_mc_samples=n_mc_hstar)

    mise_list, curves_list, beta_list, seeds = [], [], [], []
    print(f"\nRunning Ours | K={K}, n={n_samples}, noise={noise_std}, seeds={first_seed}-{first_seed+K-1}")

    for k in range(K):
        seed = first_seed + k
        seeds.append(seed)
        np.random.seed(seed)

        # Simulate one synthetic dataset (X covariates, T treatment, Y outcome).
        X, T, Y = generate_unified_data(n_samples=n_samples, noise_std=noise_std)
        h_hat, beta_sel = run_single_ours_synthetic(
            X,
            T,
            Y,
            T_GRID,
            n_samples=n_samples,
            beta0_for_f_cand=OURS_BETA0_F,
            beta_h_grid_for_h_cand=OURS_BETA_H_GRID,
            beta0_prime_for_f_tilde=OURS_BETA0_PRIME_F,
            nu_nd=OURS_NU_ND,
            nu_1d_h=OURS_NU_1D_H,
            length_scale_nd=OURS_LENGTH_SCALE_ND,
            length_scale_1d_h=OURS_LENGTH_SCALE_1D_H,
            kernel_type_nd=OURS_KERNEL_TYPE_ND,
            t_min=T_MIN,
            t_max=T_MAX,
        )

        curves_list.append(h_hat)
        beta_list.append(beta_sel)

        # Grid-averaged squared error (MISE approximation).
        mise = float(np.mean((h_hat - h_star_vals) ** 2))
        mise_list.append(mise)
        print(f"run {k+1:03d}/{K} | seed={seed} | MISE={mise:.6f} | beta={beta_sel:.6g}")

    mise_arr = np.array(mise_list, dtype=float)
    curves_mat = np.vstack(curves_list)

    # Aggregate curves across runs.
    mean_curve = curves_mat.mean(axis=0)
    std_curve = curves_mat.std(axis=0, ddof=1) if K > 1 else np.zeros_like(mean_curve)
    se_curve = std_curve / np.sqrt(K) if K > 1 else np.zeros_like(mean_curve)

    beta_arr = np.array(beta_list, dtype=float)
    beta_mean = float(beta_arr.mean())
    beta_std = float(beta_arr.std(ddof=1)) if K > 1 else 0.0
    beta_se = float(beta_std / np.sqrt(K)) if K > 1 else 0.0

    print("\n" + "="*40)
    print("Summary (Ours)")
    print("="*40)
    print(f"Mean MISE : {mise_arr.mean():.6f}")
    if K > 1:
        print(f"Std MISE  : {mise_arr.std(ddof=1):.6f}")
        print(f"SE MISE   : {mise_arr.std(ddof=1)/np.sqrt(K):.6f}")
    print("-"*40)
    print(f"Mean beta : {beta_mean:.6g}")
    if K > 1:
        print(f"Std beta  : {beta_std:.6g}")
        print(f"SE beta   : {beta_se:.6g}")

    run_tag = f"{prefix}_sample_{n_samples}_noise_{noise_std}_seeds_{seeds[0]}-{seeds[-1]}"
    return {
        "seeds": seeds,
        "t_grid": T_GRID,
        "h_star": h_star_vals,
        "mise_all": mise_arr,
        "curves_mat": curves_mat,
        "mean_curve": mean_curve,
        "se_curve": se_curve,
        "beta_selected_all": beta_arr,
        "beta_selected_mean": beta_mean,
        "beta_selected_se": beta_se,
        "run_tag": run_tag,
    }


In [4]:
# --- Plug-in (joint-kernel plug-in) ---
PLUGIN_KERNEL_TYPE_ND = "matern"
PLUGIN_LENGTH_SCALE_ND = 3.0
PLUGIN_NU_ND = 1.5

PLUGIN_C_VAL = 0.1
PLUGIN_BETA_GRID = np.array([PLUGIN_C_VAL * (2**i) for i in range(0, 9)], dtype=float)

from KRR_methods.algorithms.estimators_plugin import run_single_plugin_synthetic

def run_plugin(K=100, n_samples=1000, noise_std=1.0, first_seed=1, n_mc_hstar=None, prefix="plugin_synth"):
    """Run K Monte Carlo replications of the plug-in joint-kernel baseline.

    Args:
        K (int): Number of Monte Carlo runs.
        n_samples (int): Sample size per run.
        noise_std (float): Standard deviation of additive outcome noise.
        first_seed (int): RNG seed for the first run (subsequent runs increment).
        n_mc_hstar (int or None): MC samples for approximating h*(t).
        prefix (str): Label used in output filenames.

    Returns:
        dict: Curves, MISE values, and summary statistics.
    """
    h_star_vals = compute_h_star(T_GRID, n_mc_samples=n_mc_hstar)

    mise_list, curves_list, beta_list, seeds = [], [], [], []
    print(f"\nRunning Plug-in | K={K}, n={n_samples}, noise={noise_std}, seeds={first_seed}-{first_seed+K-1}")

    for k in range(K):
        seed = first_seed + k
        seeds.append(seed)
        np.random.seed(seed)

        X, T, Y = generate_unified_data(n_samples, noise_std)
        h_hat_vals, beta_sel = run_single_plugin_synthetic(
            X,
            T,
            Y,
            T_GRID,
            beta_grid_for_f=PLUGIN_BETA_GRID,
            nu_nd=PLUGIN_NU_ND,
            length_scale_nd=PLUGIN_LENGTH_SCALE_ND,
            kernel_type_nd=PLUGIN_KERNEL_TYPE_ND,
        )

        curves_list.append(h_hat_vals)
        beta_list.append(beta_sel)

        mise = float(np.mean((h_hat_vals - h_star_vals) ** 2))
        mise_list.append(mise)
        print(f"run {k+1:03d}/{K} | seed={seed} | MISE={mise:.6f} | beta={beta_sel:.6g}")

    mise_arr = np.array(mise_list, dtype=float)
    curves_mat = np.vstack(curves_list)

    mean_curve = curves_mat.mean(axis=0)
    std_curve = curves_mat.std(axis=0, ddof=1) if K > 1 else np.zeros_like(mean_curve)
    se_curve = std_curve / np.sqrt(K) if K > 1 else np.zeros_like(mean_curve)

    beta_arr = np.array(beta_list, dtype=float)
    beta_mean = float(beta_arr.mean())
    beta_std = float(beta_arr.std(ddof=1)) if K > 1 else 0.0
    beta_se = float(beta_std / np.sqrt(K)) if K > 1 else 0.0

    print("\n" + "="*30)
    print("Summary (Plug-in)")
    print("="*30)
    print(f"Mean MISE : {mise_arr.mean():.6f}")
    if K > 1:
        print(f"Std MISE  : {mise_arr.std(ddof=1):.6f}")
        print(f"SE MISE   : {mise_arr.std(ddof=1)/np.sqrt(K):.6f}")
    print("-"*30)
    print(f"Mean beta : {beta_mean:.6g}")
    if K > 1:
        print(f"Std beta  : {beta_std:.6g}")
        print(f"SE beta   : {beta_se:.6g}")

    run_tag = f"{prefix}_sample_{n_samples}_noise_{noise_std}_seeds_{seeds[0]}-{seeds[-1]}"
    return {
        "seeds": seeds,
        "t_grid": T_GRID,
        "h_star": h_star_vals,
        "mise_all": mise_arr,
        "curves_mat": curves_mat,
        "mean_curve": mean_curve,
        "se_curve": se_curve,
        "beta_selected_all": beta_arr,
        "beta_selected_mean": beta_mean,
        "beta_selected_se": beta_se,
        "run_tag": run_tag,
    }


In [5]:
# --- Direct regression baseline (T-only 1D KRR; ignore X) ---
from KRR_methods.algorithms.estimators_direct import (
    run_single_direct_synthetic,
    DIRECT_ELL_T,
    DIRECT_C_VAL,
    DIRECT_BETA_GRID,
)

def run_direct(K=100, n_samples=1000, noise_std=1.0, first_seed=1, n_mc_hstar=None, prefix="direct_synth"):
    """Run K Monte Carlo replications of the T-only KRR baseline.

    Args:
        K (int): Number of Monte Carlo runs.
        n_samples (int): Sample size per run.
        noise_std (float): Standard deviation of additive outcome noise.
        first_seed (int): RNG seed for the first run (subsequent runs increment).
        n_mc_hstar (int or None): MC samples for approximating h*(t).
        prefix (str): Label used in output filenames.

    Returns:
        dict: Curves, MISE values, and summary statistics.
    """
    h_star_vals = compute_h_star(T_GRID, n_mc_samples=n_mc_hstar)

    mise_list, curves_list, beta_list, seeds = [], [], [], []
    print(f"\nRunning Direct(T-only) | K={K}, n={n_samples}, noise={noise_std}, seeds={first_seed}-{first_seed+K-1}")

    for k in range(K):
        seed = first_seed + k
        seeds.append(seed)
        np.random.seed(seed)

        X, T, Y = generate_unified_data(n_samples, noise_std)
        h_hat_vals, beta_sel = run_single_direct_synthetic(
            X,
            T,
            Y,
            T_GRID,
            ell=DIRECT_ELL_T,
            beta_grid=DIRECT_BETA_GRID,
        )

        curves_list.append(h_hat_vals)
        beta_list.append(beta_sel)

        # MISE on the evaluation grid.
        mise = float(np.mean((h_hat_vals - h_star_vals) ** 2))
        mise_list.append(mise)
        print(f"run {k+1:03d}/{K} | seed={seed} | MISE={mise:.6f} | beta={beta_sel:.6g}")

    mise_arr = np.array(mise_list, dtype=float)
    curves_mat = np.vstack(curves_list)

    mean_curve = curves_mat.mean(axis=0)
    std_curve = curves_mat.std(axis=0, ddof=1) if K > 1 else np.zeros_like(mean_curve)
    se_curve = std_curve / np.sqrt(K) if K > 1 else np.zeros_like(mean_curve)

    beta_arr = np.array(beta_list, dtype=float)
    beta_mean = float(beta_arr.mean())
    beta_std = float(beta_arr.std(ddof=1)) if K > 1 else 0.0
    beta_se = float(beta_std / np.sqrt(K)) if K > 1 else 0.0

    print("\n" + "="*30)
    print("Summary (Direct T-only)")
    print("="*30)
    print(f"Mean MISE : {mise_arr.mean():.6f}")
    if K > 1:
        print(f"Std MISE  : {mise_arr.std(ddof=1):.6f}")
        print(f"SE MISE   : {mise_arr.std(ddof=1)/np.sqrt(K):.6f}")
    print("-"*30)
    print(f"Mean beta : {beta_mean:.6g}")
    if K > 1:
        print(f"Std beta  : {beta_std:.6g}")
        print(f"SE beta   : {beta_se:.6g}")

    run_tag = f"{prefix}_sample_{n_samples}_noise_{noise_std}_seeds_{seeds[0]}-{seeds[-1]}"
    return {
        "seeds": seeds,
        "t_grid": T_GRID,
        "h_star": h_star_vals,
        "mise_all": mise_arr,
        "curves_mat": curves_mat,
        "mean_curve": mean_curve,
        "se_curve": se_curve,
        "beta_selected_all": beta_arr,
        "beta_selected_mean": beta_mean,
        "beta_selected_se": beta_se,
        "run_tag": run_tag,
    }


## Run selector

Set the parameters below and run the cell to execute one algorithm and save CSV outputs.


In [6]:
# --- Selector (no widgets) ---

def dispatch_run(algo, n_samples, K, noise_std, first_seed, save_csv=True):
    """Dispatch to a specific estimator by name.

    Args:
        algo (str): One of "ours", "plugin", or "direct" (case-insensitive).
        n_samples (int): Sample size per run.
        K (int): Number of Monte Carlo runs.
        noise_std (float): Standard deviation of additive outcome noise.
        first_seed (int): RNG seed for the first run.
        save_csv (bool): If True, save per-seed curves and summaries to CSV.

    Returns:
        dict: Results dictionary returned by the selected run_* function.
    """
    algo = str(algo).lower().strip()
    if algo in ["ours", "joint-ours", "joint_ours", "ours (joint)"]:
        results = run_ours(K=K, n_samples=n_samples, noise_std=noise_std, first_seed=first_seed)
        pretty = f"Ours | n={n_samples} | K={K} | noise={noise_std}"
    elif algo in ["plugin", "plug-in", "plug_in"]:
        results = run_plugin(K=K, n_samples=n_samples, noise_std=noise_std, first_seed=first_seed)
        pretty = f"Plug-in | n={n_samples} | K={K} | noise={noise_std}"
    elif algo in ["direct", "direct-regression", "direct regression", "t-only", "t_only"]:
        results = run_direct(K=K, n_samples=n_samples, noise_std=noise_std, first_seed=first_seed)
        pretty = f"Direct(T-only) | n={n_samples} | K={K} | noise={noise_std}"
    else:
        raise ValueError(f"Unknown algo: {algo}")

    if save_csv:
        save_results_csv(results, prefix=algo)

    return results


In [8]:
ALGO = "direct"     # "ours" | "plugin" | "direct"
N_SAMPLES = 500     # Sample size per Monte Carlo run
K_RUNS = 100        # Number of Monte Carlo runs
NOISE_STD = 1.0     # Outcome noise standard deviation
FIRST_SEED = 0      # RNG seed for the first run

results = dispatch_run(
    algo=ALGO,
    n_samples=N_SAMPLES,
    K=K_RUNS,
    noise_std=NOISE_STD,
    first_seed=FIRST_SEED,
    save_csv=True
)
print("run_tag =", results["run_tag"])



🚀 Running Direct(T-only) | K=100, n=500, noise=1.0, seeds=0-99
  - run 001/100 | seed=0 | MISE=0.138969 | beta=0.4 | valMSE=1.047108
  - run 002/100 | seed=1 | MISE=0.132085 | beta=6.4 | valMSE=0.955397
  - run 003/100 | seed=2 | MISE=0.117171 | beta=0.8 | valMSE=0.987428
  - run 004/100 | seed=3 | MISE=0.119227 | beta=3.2 | valMSE=1.036638
  - run 005/100 | seed=4 | MISE=0.218955 | beta=0.1 | valMSE=1.107549
  - run 006/100 | seed=5 | MISE=0.130775 | beta=0.8 | valMSE=1.001766
  - run 007/100 | seed=6 | MISE=0.120388 | beta=0.1 | valMSE=1.095198
  - run 008/100 | seed=7 | MISE=0.153747 | beta=1.6 | valMSE=0.903965
  - run 009/100 | seed=8 | MISE=0.170643 | beta=3.2 | valMSE=0.841739
  - run 010/100 | seed=9 | MISE=0.187686 | beta=3.2 | valMSE=1.070032
  - run 011/100 | seed=10 | MISE=0.137322 | beta=6.4 | valMSE=0.898516
  - run 012/100 | seed=11 | MISE=0.121189 | beta=6.4 | valMSE=1.099859
  - run 013/100 | seed=12 | MISE=0.131960 | beta=3.2 | valMSE=0.960426
  - run 014/100 | seed=

/tmp/ipython-input-465978959.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f"h_hat_seed_{seed}"] = results["curves_mat"][i, :]
/tmp/ipython-input-465978959.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out["mean_h_hat"] = results["mean_curve"]
/tmp/ipython-input-465978959.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-f

✅ Saved: /content/drive/MyDrive/Colab Notebooks/CTE_Codes/KRR_methods/Results/direct_synth_sample_500_noise_1.0_seeds_0-99.csv
run_tag = direct_synth_sample_500_noise_1.0_seeds_0-99
